# CS336 Lecture 01 Notes: Tokenization and BPE

Source: https://cs336.stanford.edu/lectures/?trace=lecture_01&step=1

這份 notebook 是 Lecture 01 程式例子的整理版，重點放在課堂後半段的 tokenization。原網頁是 executable lecture / trace viewer，裡面完整的 Python source 會一邊執行一邊顯示變數。這裡改成比較適合你自己在 JupyterLab 裡逐格執行、修改、實驗的格式。

## 這堂課的程式主線

Language model 不直接處理 Python 字串，而是處理一串整數 token IDs。所以 tokenizer 要做兩件事：

1. `encode(string) -> list[int]`: 把文字轉成 token IDs。
2. `decode(indices) -> str`: 把 token IDs 轉回文字。

一個好的 tokenizer 通常需要能 round-trip：

```python
decode(encode(text)) == text
```

Lecture 01 依序比較：character tokenizer、byte tokenizer、word tokenizer，最後進到 Byte Pair Encoding, BPE。

In [1]:
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
import re


class Tokenizer(ABC):
    """Minimal tokenizer interface used in the lecture."""

    @abstractmethod
    def encode(self, string: str) -> list[int]:
        raise NotImplementedError

    @abstractmethod
    def decode(self, indices: list[int]) -> str:
        raise NotImplementedError


def get_compression_ratio(string: str, indices: list[int]) -> float:
    """Return UTF-8 bytes per token."""
    num_bytes = len(string.encode("utf-8"))
    num_tokens = len(indices)
    return num_bytes / num_tokens

## 1. Character tokenizer

最直覺的方法：每個 Unicode character 變成一個整數。Python 裡 `ord()` 可以把字元轉成 code point，`chr()` 可以轉回來。

In [2]:
class CharacterTokenizer(Tokenizer):
    """Represent a string as Unicode code points."""

    def encode(self, string: str) -> list[int]:
        return [ord(ch) for ch in string]

    def decode(self, indices: list[int]) -> str:
        return "".join(chr(i) for i in indices)


string = "Hello, 🌍! 你好!"
char_tokenizer = CharacterTokenizer()
char_indices = char_tokenizer.encode(string)
char_reconstructed = char_tokenizer.decode(char_indices)

print(char_indices)
print(char_reconstructed)
print("round trip:", string == char_reconstructed)
print("compression ratio:", get_compression_ratio(string, char_indices), f"({len(string.encode('utf-8'))}/{len(char_indices)})")

[72, 101, 108, 108, 111, 44, 32, 127757, 33, 32, 20320, 22909, 33]
Hello, 🌍! 你好!
round trip: True
compression ratio: 1.5384615384615385 (20/13)


觀察：character tokenizer 可以 round-trip，但 vocab 可能很大，因為 Unicode code point 很多，而且很多字元很少見。

## 2. Byte tokenizer

另一個方法：先把字串用 UTF-8 變成 bytes。每個 byte 都是 0 到 255，所以 vocabulary 固定只有 256 個。

In [3]:
class ByteTokenizer(Tokenizer):
    """Represent a string as UTF-8 bytes."""

    def encode(self, string: str) -> list[int]:
        return list(string.encode("utf-8"))

    def decode(self, indices: list[int]) -> str:
        return bytes(indices).decode("utf-8")


byte_tokenizer = ByteTokenizer()
byte_indices = byte_tokenizer.encode(string)
byte_reconstructed = byte_tokenizer.decode(byte_indices)

print(byte_indices)
print(byte_reconstructed)
print("round trip:", string == byte_reconstructed)
print("vocab size:", 256)
print("compression ratio:", get_compression_ratio(string, byte_indices), f"({len(string.encode('utf-8'))}/{len(byte_indices)})")

[72, 101, 108, 108, 111, 44, 32, 240, 159, 140, 141, 33, 32, 228, 189, 160, 229, 165, 189, 33]
Hello, 🌍! 你好!
round trip: True
vocab size: 256
compression ratio: 1.0 (20/20)


觀察：byte tokenizer 的 vocab 很小，但序列很長。對 Transformer 來說，序列太長會很貴，因為 attention 對 sequence length 大致是 quadratic。

## 3. Word-ish tokenizer

傳統 NLP 常會把文字切成接近 word 的 chunk。這讓 token 比較有語意，但會遇到 vocabulary 太大、罕見字、沒看過的新字等問題。

In [4]:
word_string = "I'll say supercalifragilisticexpialidocious!"

# Similar spirit to the lecture example: keep word characters together,
# and keep every non-space punctuation as its own chunk.
chunks = re.findall(r"\w+|.", word_string)

print(chunks)
print("compression ratio:", get_compression_ratio(word_string, chunks), f"({len(word_string.encode('utf-8'))}/{len(chunks)})")

['I', "'", 'll', ' ', 'say', ' ', 'supercalifragilisticexpialidocious', '!']
compression ratio: 5.5 (44/8)


觀察：word-based tokenization 的 compression ratio 看起來不錯，但真實訓練資料裡 distinct words / chunks 可能非常多，而且測試時可能遇到訓練沒看過的字。

## 4. Byte Pair Encoding, BPE

BPE 的核心想法：

1. 一開始每個 byte 都是一個 token。
2. 統計相鄰 token pair 出現次數。
3. 把最常見的 pair merge 成一個新的 token。
4. 重複多次。

這樣常見片段會被壓成比較短的 token 序列，罕見片段仍然可以退回 byte-level 表示，因此不需要 UNK token。

In [5]:
def count_adjacent_pairs(indices: list[int]) -> dict[tuple[int, int], int]:
    """Count adjacent token pairs."""
    counts = defaultdict(int)
    for pair in zip(indices, indices[1:]):
        counts[pair] += 1
    return dict(counts)


def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:
    """Replace all occurrences of pair with new_index."""
    new_indices = []
    i = 0
    while i < len(indices):
        if i + 1 < len(indices) and (indices[i], indices[i + 1]) == pair:
            new_indices.append(new_index)
            i += 2
        else:
            new_indices.append(indices[i])
            i += 1
    return new_indices


@dataclass(frozen=True)
class BPETokenizerParams:
    vocab: dict[int, bytes]
    merges: dict[tuple[int, int], int]


def train_bpe(string: str, num_merges: int) -> BPETokenizerParams:
    indices = list(string.encode("utf-8"))
    vocab = {i: bytes([i]) for i in range(256)} # token_id -> bytes
    merges = {} # pair -> new_token_id

    print("start:", indices, bytes(indices))
    for step in range(num_merges):
        counts = count_adjacent_pairs(indices)
        pair = max(counts, key=counts.get)
        new_index = 256 + step

        merges[pair] = new_index
        vocab[new_index] = vocab[pair[0]] + vocab[pair[1]]
        indices = merge(indices, pair, new_index)

        print(f"merge {step + 1}: pair={pair} -> {new_index}, token={vocab[new_index]!r}") # repr()
        print("        indices:", indices)

    print("final compression ratio:", get_compression_ratio(string, indices), f"({len(string.encode('utf-8'))}/{len(indices)})")
    return BPETokenizerParams(vocab=vocab, merges=merges)

In [6]:
params = train_bpe("the cat in the hat", num_merges=3)

start: [116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116] b'the cat in the hat'
merge 1: pair=(116, 104) -> 256, token=b'th'
        indices: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116]
merge 2: pair=(256, 101) -> 257, token=b'the'
        indices: [257, 32, 99, 97, 116, 32, 105, 110, 32, 257, 32, 104, 97, 116]
merge 3: pair=(257, 32) -> 258, token=b'the '
        indices: [258, 99, 97, 116, 32, 105, 110, 32, 258, 104, 97, 116]
final compression ratio: 1.5 (18/12)


Lecture 裡的 toy example 會先學到 `t` + `h`，再學到 `th` + `e` 這類常見片段。真實 tokenizer 會在非常大的 corpus 上做很多次 merge。

In [7]:
class BPETokenizer(Tokenizer):
    """Simple BPE tokenizer using a fixed list of learned merges."""

    def __init__(self, params: BPETokenizerParams):
        self.params = params

    def encode(self, string: str) -> list[int]:
        indices = list(string.encode("utf-8"))

        # This is intentionally simple and slow: apply merges in training order.
        # Python 3.7 之後，dict 會保留插入順序。
        for pair, new_index in self.params.merges.items():
            indices = merge(indices, pair, new_index)

        return indices

    def decode(self, indices: list[int]) -> str:
        byte_chunks = [self.params.vocab[i] for i in indices]
        return b"".join(byte_chunks).decode("utf-8")


bpe = BPETokenizer(params)
new_string = "the quick brown fox"
bpe_indices = bpe.encode(new_string)
bpe_reconstructed = bpe.decode(bpe_indices)

print(bpe_indices)
print(bpe_reconstructed)
print("round trip:", new_string == bpe_reconstructed)
print("compression ratio:", get_compression_ratio(new_string, bpe_indices), f"({len(new_string.encode('utf-8'))}/{len(bpe_indices)})")

[258, 113, 117, 105, 99, 107, 32, 98, 114, 111, 119, 110, 32, 102, 111, 120]
the quick brown fox
round trip: True
compression ratio: 1.1875 (19/16)


## Optional: try OpenAI/tiktoken

Lecture 01 也示範了真實 tokenizer。這格需要先安裝 `tiktoken`：

```bash
python -m pip install tiktoken
```

如果還沒裝，下面這格會印出提醒，不會讓整份 notebook 壞掉。

In [9]:
try:
    import tiktoken

    tokenizer = tiktoken.get_encoding("o200k_base")
    sample = "Hello, 🌍! 你好!"
    ids = tokenizer.encode(sample)
    decoded = tokenizer.decode(ids)

    print(ids)
    print(decoded)
    print("round trip:", sample == decoded)
    print("vocab size:", tokenizer.n_vocab)
    print("compression ratio:", get_compression_ratio(sample, ids), f"({len(sample.encode('utf-8'))}/{len(ids)})")
except ModuleNotFoundError:
    print("tiktoken is not installed yet. Run: python -m pip install tiktoken")

[13225, 11, 130321, 235, 0, 220, 177519, 0]
Hello, 🌍! 你好!
round trip: True
vocab size: 200019
compression ratio: 2.5 (20/8)


## 小練習

1. 把 `train_bpe("the cat in the hat", num_merges=3)` 改成 `num_merges=5`，觀察 indices 怎麼變短。
2. 把訓練字串改成 `"banana bandana banana"`，看最常被 merge 的 pair 是什麼。
3. 用同一個 BPE tokenizer encode 一句沒有在訓練字串裡出現過的話，確認它仍然可以 round-trip。
4. 比較 character / byte / BPE 對中文、emoji、英文長字的 compression ratio。
5. 思考 Assignment 1 為什麼不能用這個 naive BPE：`encode()` 每次都掃過所有 merges，真實 vocab 可能有十萬級別的 merges。

## Takeaways

- Tokenizer 是文字和 token IDs 之間的轉換器。
- Character tokenizer round-trip 容易，但 vocabulary 和稀疏性不理想。
- Byte tokenizer vocabulary 很小，但 sequence 太長。
- Word tokenizer 語意直覺好，但 vocabulary 無界且有 OOV 問題。
- BPE 從 bytes 出發，透過資料驅動的 merges，把常見片段壓成短序列，同時保留處理任何文字的能力。